<a href="https://colab.research.google.com/github/Existanze54/sirius-machine-learning-2026/blob/main/Seminars/S7_DimRed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Семинар 7. Снижение размерности

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import scale
from sklearn.decomposition import PCA

### Задача 1. Визуализация транскриптомных данных

Загрузим транскритомный датасет и метаданные к нему.

In [2]:
! wget -q https://data.bioml.ru/htdocs/courses/bioml/classic_ml/unsupervised/dim_reduction/data/gse53625.tar.gz -O gse53625.tar.gz
! tar xvzf gse53625.tar.gz

"wget" ­Ґ пў«пҐвбп ў­гваҐ­­Ґ© Ё«Ё ў­Ґи­Ґ©
Є®¬ ­¤®©, ЁбЇ®«­пҐ¬®© Їа®Ја ¬¬®© Ё«Ё Ї ЄҐв­л¬ д ©«®¬.
tar: Error opening archive: Failed to open 'gse53625.tar.gz'


In [3]:
df = pd.read_csv('data/gse53625_expression.csv', index_col=0)
mdf = pd.read_csv('data/gse53625_metadata.csv', index_col=0)

FileNotFoundError: [Errno 2] No such file or directory: 'data/gse53625_expression.csv'

In [ ]:
df.head(2)

In [ ]:
mdf.head(2)

In [ ]:
X = df.T
mark = mdf['Sample type'] == 'cancer'

Обучим и применим PCA.

In [ ]:
pca = PCA(n_components=2) # по умолчанию число компонент = min(число признаков, число наблюдений)
drX = pca.fit_transform(X)

Визуализируем.

In [ ]:
x = drX[:, 0]
y = drX[:, 1]

plt.scatter(x[~mark], y[~mark])
plt.scatter(x[mark], y[mark])

plt.legend(['normal', 'cancer'])
plt.show()

Что мы забыли сделать перед PCA?

In [ ]:
# your code here

### Задача 2. Коррекция эффекта

Что в метаданных может это объяснить?

In [ ]:
mdf.describe()

In [ ]:
X = df.T
mark1 = mdf['Sample type']
mark2 = mdf['Dataset']
drX = pca.fit_transform(X)

In [ ]:
data = pd.DataFrame({
    'PC1': drX[:, 0],
    'PC2': drX[:, 1],
    'phenotype': mark1,
    'batch': mark2,
})

In [ ]:
sns.scatterplot(data, x='PC1', y='PC2',
                hue='phenotype', style='batch')
plt.show()

Используем специализированную библиотеку [pyComBat](https://github.com/epigenelabs/pyComBat) для коррекции batch-эффекта.

In [ ]:
! pip -q install combat

In [ ]:
from combat.pycombat import pycombat
df = pycombat(df, mdf['Dataset'])

In [ ]:
X = df.T
mark1 = mdf['Sample type']
mark2 = mdf['Dataset']
drX = pca.fit_transform(X)

А теперь давайте сделаем все по учебнику, со стандартизацией.

In [ ]:
# your code here

### Задача 3. Предобработка признаков

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import Normalizer
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

def train_test_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    return pd.DataFrame({'MAE': [mae], 'R2': [r2]})

In [ ]:
! wget -q https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/SAMPL.csv

In [ ]:
df = pd.read_csv('SAMPL.csv')
smis = df['smiles']
y = df['expt']

In [ ]:
! pip -q install datamol

In [ ]:
import datamol as dm

X = np.array([dm.to_fp(smi, fp_type='ecfp') for smi in smis])
X

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=0)

In [ ]:
model = RidgeCV()
train_test_model(model, X_train, X_test, y_train, y_test)

In [ ]:
pipe = Pipeline([
    ("reducer", PCA()),
    ("scaler", Normalizer()),
    ("model", RidgeCV()),
])
train_test_model(pipe, X_train, X_test, y_train, y_test)